In [1]:
import json
import os
import chromadb
from typing import Annotated, TYPE_CHECKING

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent,FunctionResultContent, StreamingTextContent, ChatMessageContent
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.functions import kernel_function

from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents import AuthorRole

if TYPE_CHECKING:
    from chromadb.api.models.Collection import Collection
# Initialize the asynchronous OpenAI client
from dotenv import load_dotenv


In [2]:
from typing import Annotated, Dict, Any, List
from semantic_kernel.functions import kernel_function

In [3]:
import json
from typing import Dict, Any

class FlairPlugin:

    transcriptome_description = (
        "This module generates a transcriptome of high confidence isoforms (bed, gtf, and fasta files) "
        "directly from a BAM file of aligned reads. This is 3x faster and uses 20x less memory than correct + collapse.\n\n"
        "To get aligned reads, you can use FLAIR align, or just run the following command to generate the BAM file to use as input\n"
        "If you want to run downstream fusion detection with FLAIR fusion, run flair align with –filtertype separate to generate "
        "a separate file of chimeric alignments.\n\n"
        "This module does not currently have all of the options included in collapse, such as promoter/3’ end filtering. "
        "Other options have been simplified or combined. For instance, the collapse –annotation_reliant option from flair "
        "collapse is now the default. To run without relying on annotation as strongly, specify –noaligntoannot."
    )

    transcriptome_parameters = {
        "-o/--output": "output file name base for FLAIR isoforms (default: flair.collapse)",
        "-t/--threads": "minimap2 number of threads (4)",
        "-f/--gtf": "[HIGHLY RECOMMENDED] GTF annotation file, used for renaming FLAIR isoforms to annotated isoforms and adjusting TSS/TESs",
        "--stringent": "[HIGHLY RECOMMENDED] specify if all supporting reads need to be full-length (spanning 25 bp of the first and last exons)",
        "--check_splice": "[HIGHLY RECOMMENDED] enforce coverage of 4 out of 6 bp around each splice site and no insertions greater than 3 bp at the splice site. DON'T USE WITH DATA WITH HIGH ERROR RATES (old direct-RNA)",
        "--noaligntoannot": "related to old annotation_reliant, now specify if you don't want an initial alignment to the annotated sequences and only want transcript detection from the genomic alignment. Will be slightly faster but less accurate if the annotation is good",
        "-n/--no_redundant": "For each unique splice junction chain, report options include: none--best TSSs/TESs chosen for each unique set of splice junctions; longest--single TSS/TES chosen to maximize length; best_only--single most supported TSS/TES used in conjunction chosen (none)",
        "--filter": "Report options include: default--subset isoforms are removed based on support; nosubset--any isoforms that are a proper set of another isoform are removed; comprehensive--default set + all subset isoforms; ginormous--comprehensive set + single exon subset isoforms",
        "--splittoregion": "force running on each region of non-overlapping reads, no matter the file size. default: parallelize by chromosome if file is <1G, otherwise parallelize on all regions of non-overlapping reads",
        "--predictCDS": "specify if you want to predict the CDS of the final isoforms. Will be output in the final bed file but not the gtf file. Productivity annotation is also added in the name field, which is detailed further in the predictProductivity documentation"
    }

    @kernel_function(
        description="This module generates a transcriptome of high confidence isoforms (bed, gtf, and fasta files).",
        name="flair_transcriptome"
    )
    def flair_transcriptome(
        self, 
        user_task: Annotated[str, "Brief description of the user's analysis goal"],
        bam_file: Annotated[str, "Required: Path to the aligned BAM file for processing"],
        genome: Annotated[str, "Required: Path to the reference genome FASTA file"]
    ) -> str:
        prompt = (
            "You are an expert bioinformatician skilled in using FLair to annotate transcriptomes. "
            "You will be given a user task, a BAM file, and a GTF file. "
            "Your goal is to generate accurate codes that fulfills the user's request."
            "Easiest usage (Only required parameters): flair transcriptome -b reads.genomealigned.bam -g genome.fa"
        )

        prompt += f"User Task: {user_task}\n\n"
        prompt += "Required Parameters:\n"
        prompt += f"BAM File: {bam_file}\n\n"
        prompt += f"Genome: {genome}\n\n"

        prompt += "Optional Parameters:\n"
        for param, desc in self.transcriptome_parameters.items():
            prompt += f"{param}: {desc}\n"

        prompt += str(self.transcriptome_description)

        return prompt

    @kernel_function(
        description="This module identifes the best isoform assignment based on alignment quality, fraction of read aligned, and fraction of transcript aligned. Then quantifies the expression of each isoform.",
        name="flair_quantify"
    )
    def flair_quantify(
        self, 
        user_task: Annotated[str, "Description of each sample and containing sample id, condition, batch"],
        isoforms_fa: Annotated[str, "Required: Path to the isoforms FASTA file"],
    ) -> str:
        prompt = (
            "You are an expert bioinformatician skilled in using FLair to quantify the expression of each isoform. "
            "You will be given a user task and an isoforms fasta file. "
            "First, create a reads_manifest: a tab-delimited file listing sample id, condition, batch, and the path to the corresponding sample fastq file (reads.fq) for each sample called reads_manifest.tsv"
            "Your goal is to generate accurate codes that fulfills the user's request."
            "Easiest usage (Only required parameters): flair quantify -i isoforms.fa -r reads_manifest.tsv"
        )

        prompt += f"Isoform Fasta: {isoforms_fa}\n\n"
        prompt += f"User Sample Id, Condition, Batch: {user_task}\n\n"


        return prompt

In [4]:
# ------------------------------
# 1️⃣ 初始化 OpenAI 客户端
# ------------------------------
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

In [5]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "FlairAgent"

AGENT_INSTRUCTIONS = """You are an planner agent.
    Your job is to decide which required parameters to run remembering that the user's request is to run the flair software.
    Below are the available agents specialised in different tasks:
    - flair_transcriptome: This subcommand generates a transcriptome of high confidence isoforms (bed, gtf, and fasta files) directly from a BAM file of aligned reads.
    Required Input: user_task, bam_file, genome
    - flair_quantify: Default: identifes the best isoform assignment based on alignment quality, fraction of read aligned, and fraction of transcript aligned
    Required Input: user_task, reads_manifest.tsv, isoforms.fa

    Your output show show each required parameter and the value you chose.
    If the user does not provide enought required parameters, you should ask again.
"""

agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an planner agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
)

In [6]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "CodeGeneratorAgent"

AGENT_INSTRUCTIONS = """You are an code generator agent.
    Your job is to generate the code based on the user's request and the subcommand parameter description.
    Only generate code if user provides all required parameters.
    You should follow the steps:
    - Generate the code prompt based on the user's request and the subcommand parameter description.
    - Generate the code based on the code prompt in step 1.
    - Explain the parameter in the code.

    Important:
    - Your code section should start with <code> and end with </code>.
    - Your code MUST start with 'flair '
"""

code_agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an code generator agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    plugins=[FlairPlugin()],
)

In [7]:
from semantic_kernel.agents import SequentialOrchestration, GroupChatOrchestration, RoundRobinGroupChatManager

from semantic_kernel.contents import ChatMessageContent

def agent_response_callback(message: ChatMessageContent) -> None:
    print(f"# {message.name}\n{message.content}")




In [8]:
from semantic_kernel.contents import ChatHistorySummarizationReducer
from semantic_kernel.agents import SequentialOrchestration, GroupChatOrchestration, RoundRobinGroupChatManager
# Configure reduction parameters
REDUCER_TARGET_COUNT = 1  # Target number of messages to keep after reduction
REDUCER_THRESHOLD =  4 # Trigger reduction when message count exceeds this

history_reducer = ChatHistorySummarizationReducer(
    service=chat_completion_service,
    target_count=REDUCER_TARGET_COUNT,
    threshold_count=REDUCER_THRESHOLD,
)

chat = SequentialOrchestration(
    members=[agent, code_agent],
    agent_response_callback=agent_response_callback,
)

# GroupChatOrchestration(
#     members=[agent, code_agent],
#     manager=RoundRobinGroupChatManager(max_rounds=5),  # Odd number so writer gets the last word
#     agent_response_callback=agent_response_callback,
# )

In [9]:
from semantic_kernel.agents.runtime import InProcessRuntime
runtime = InProcessRuntime()
runtime.start()

# FlairAgent
To run the flair_transcriptome command for assembling the transcriptome, I need the following required parameters:

1. **user_task**: This needs to be specified. (e.g., "assemble_transcriptome")
2. **bam_file**: mybam.bam
3. **genome**: genome.fa

Please specify the **user_task** you wish to perform.
# CodeGeneratorAgent

# CodeGeneratorAgent

# CodeGeneratorAgent
Here's the code that can be used to assemble the transcriptome from the BAM file and reference genome provided:

<code>
flair transcriptome -b mybam.bam -g genome.fa -o output_file_name -t 4
</code>

### Parameter Explanation:

- **-b mybam.bam**: Specifies the BAM file containing the aligned reads. This is a required input for the command.
- **-g genome.fa**: Specifies the reference genome in FASTA format. This is also a required parameter necessary for the transcriptome assembly.
- **-o output_file_name**: This is an optional parameter for specifying the base name for the output files generated by FLAIR. If not

In [10]:
user_inputs = ["I want to assemble the transcriptome for mybam.bam and my reference genome is genome.fa",
                "Do you have any other parameters I should use you recommend?"]

async def main():
    thread = ChatHistoryAgentThread(chat_history=history_reducer)
    for user_input in user_inputs:
        history_reducer.add_user_message(user_input)
        orchestration_result = await chat.invoke(
            task=history_reducer.messages,
            runtime=runtime,
        )
        value = await orchestration_result.get(timeout=100)
        print(f"***** Final Result *****\n{value}")
        history_reducer.add_assistant_message(value.content)

        if len(thread) > 4:
            await thread.reduce()
    await runtime.stop_when_idle()

await main()

***** Final Result *****
Here's the code that can be used to assemble the transcriptome from the BAM file and reference genome provided:

<code>
flair transcriptome -b mybam.bam -g genome.fa -o output_file_name -t 4
</code>

### Parameter Explanation:

- **-b mybam.bam**: Specifies the BAM file containing the aligned reads. This is a required input for the command.
- **-g genome.fa**: Specifies the reference genome in FASTA format. This is also a required parameter necessary for the transcriptome assembly.
- **-o output_file_name**: This is an optional parameter for specifying the base name for the output files generated by FLAIR. If not provided, it defaults to "flair.collapse".
- **-t 4**: This optional parameter specifies the number of threads to use for the alignment process. By default, it is set to 4 threads, allowing parallel processing to speed up the task.

You can adjust the output file's name and the number of threads according to your requirements.
***** Final Result *****
